#  AI Lead Generation Agent

**Purpose:** Automatically research the web and produce a structured list of potential business leads (companies that match a target customer profile), ready to export as a spreadsheet.

**Cost:** $0 — built entirely on free tools, no credit card required anywhere:
- **Google Gemini API** (free tier) — the reasoning engine
- **Tavily Search API** (free tier, 1,000 searches/month) — the web search tool, purpose-built for AI agents
- **Google Colab** — free cloud notebook, nothing to install on your laptop

**How it works (in plain terms):**
1. You describe your Ideal Customer Profile (industry, location, company type, etc.)
2. The agent plans several targeted search queries on its own
3. The agent searches the live web for matching companies
4. The agent reads the results and extracts a clean, structured lead list
5. You review, refine, and export the leads to a CSV file (opens in Excel/Sheets)

This is an **agentic** workflow because the AI is not just answering one question — it plans its own search steps, uses a tool (web search) to gather live information, and then reasons over the results to produce a final output, with minimal human input at each step.

---
*Prepared by: [Abhay Sathe] — [19/08/2026]*

In [ ]:
# Step 1 — Install the free libraries we need
!pip install -q -U google-generativeai tavily-python pandas==2.2.3

In [ ]:
# Step 2 — Connect to the free Gemini API using the secret key you added above
import google.generativeai as genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# Google occasionally renames its free models, so we try a short list
# of current free Flash models and use the first one that responds.
CANDIDATE_MODELS = [
    "gemini-flash-latest",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
]

model = None
for name in CANDIDATE_MODELS:
    try:
        candidate = genai.GenerativeModel(name)
        candidate.generate_content("ping")  # quick test call
        model = candidate
        print(f"✅ Connected to Gemini using model: {name}")
        break
    except Exception as e:
        print(f"⚠️ {name} not available right now, trying next option...")

if model is None:
    raise RuntimeError(
        "Could not connect to any free Gemini model. "
        "Double-check your GEMINI_API_KEY secret in Step 1."
    )

✅ Connected to Gemini using model: gemini-flash-latest


In [ ]:
# Step 4 — Set up the free Tavily web search tool (built specifically for AI agents)
from tavily import TavilyClient

TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def web_search(query, max_results=6):
    """Searches the live web for free (within Tavily's free monthly quota)
    and returns a list of results (title, link, snippet)."""
    response = tavily_client.search(query=query, max_results=max_results)
    return response.get("results", [])

# Quick connectivity test
test = web_search("test query", max_results=1)
print(f"✅ Tavily search is working ({len(test)} result returned for test query)")

✅ Tavily search is working (1 result returned for test query)


## Step 5 — Describe your Ideal Customer Profile (ICP)

Edit the values below to match who you're trying to reach. This is the **only part you normally need to change** day-to-day.

In [ ]:
# EDIT THESE VALUES FOR YOUR CAMPAIGN
TARGET_PROFILE = {
    "industry": "boutique digital marketing agencies",
    "location": "Ahmedabad, India",
    "company_size": "5-50 employees",
    "notes": "Looking for agencies that might need a CRM/automation tool"
}

NUMBER_OF_LEADS = 10  # how many leads you want in the final list

## Step 6 — The Agent: plan → search → extract

Run the next three cells as-is. This is the "agentic" core: the AI plans its own search queries, calls the search tool, then reasons over everything it found to build the lead table.

In [ ]:
import json

def plan_search_queries(profile, n_queries=4):
    """Agent step 1: the AI decides what to search for, based on the ICP."""
    prompt = f"""
You are a B2B lead research planner.
Ideal Customer Profile: {json.dumps(profile)}

Write {n_queries} short, effective search queries that would help find real,
specific companies matching this profile. Return ONLY a JSON list of strings,
nothing else. Example: ["query one", "query two"]
"""
    response = model.generate_content(prompt)
    text = response.text.strip().strip("`").replace("json", "", 1).strip()
    try:
        return json.loads(text)
    except Exception:
        # fallback: one simple query built from the profile
        return [f"{profile.get('industry','')} companies {profile.get('location','')}"]

queries = plan_search_queries(TARGET_PROFILE)
print("🧠 The agent planned these searches:")
for q in queries:
    print(" -", q)

ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 31580.40ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3364.66ms


🧠 The agent planned these searches:
 - site:linkedin.com/company "digital marketing agency" Ahmedabad "11-50 employees"
 - "boutique digital marketing agency" Ahmedabad
 - site:clutch.co/agencies/digital-marketing Ahmedabad
 - "performance marketing agency" OR "lead generation agency" Ahmedabad


In [ ]:
# Agent step 2: run every planned search and collect the raw results
# (each query uses 1 Tavily credit — well within the 1,000/month free quota)
all_results = []
for q in queries:
    try:
        results = web_search(q, max_results=6)
        all_results.extend(results)
        print(f"🔎 '{q}' → {len(results)} results found")
    except Exception as e:
        print(f"⚠️ Search failed for '{q}': {e}")

print(f"\nTotal raw results collected: {len(all_results)}")

🔎 'site:linkedin.com/company "digital marketing agency" Ahmedabad "11-50 employees"' → 6 results found
🔎 '"boutique digital marketing agency" Ahmedabad' → 6 results found
🔎 'site:clutch.co/agencies/digital-marketing Ahmedabad' → 6 results found
🔎 '"performance marketing agency" OR "lead generation agency" Ahmedabad' → 6 results found

Total raw results collected: 24


In [ ]:
# Agent step 3: the AI reads all the raw search results and extracts a
# clean, structured lead list matching the ICP.
import pandas as pd

raw_text = "\n\n".join(
    f"Title: {r.get('title','')}\nLink: {r.get('url','')}\nSnippet: {r.get('content','')}"
    for r in all_results
)

extraction_prompt = f"""
You are a B2B lead qualification assistant.

Ideal Customer Profile: {json.dumps(TARGET_PROFILE)}

Below are raw web search results. Identify up to {NUMBER_OF_LEADS} distinct,
real COMPANIES (not directories, listicles, or news sites) that best match
the profile above.

For each one, return an object with:
- company_name
- website (best guess from the link, root domain only)
- why_a_fit (1 short sentence, specific to this company)
- suggested_next_step (1 short, practical outreach idea)

Return ONLY a JSON list of objects, nothing else — no markdown, no commentary.

SEARCH RESULTS:
{raw_text}
"""

response = model.generate_content(extraction_prompt)
cleaned = response.text.strip().strip("`").replace("json", "", 1).strip()

try:
    leads = json.loads(cleaned)
except Exception:
    print("⚠️ Could not parse the model's output as JSON. Raw output below:\n")
    print(cleaned)
    leads = []

leads_df = pd.DataFrame(leads)
leads_df

ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 7916.00ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6446.29ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 17594.11ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 9961.39ms


,company_name,website,why_a_fit,suggested_next_step
0,Mark Honest Digital Solution Pvt. Ltd,markhonestdigital.com,11-50 employee boutique agency in Ahmedabad fo...,Reach out to founder Deepak Singh on LinkedIn ...
1,Times Tag,timestag.com,Ahmedabad-based 12-member digital agency runni...,Send a direct email to Sohel Shaikh proposing ...
2,Keadigi,keadigi.com,Ahmedabad boutique agency offering performance...,Contact their agency email with a case study o...
3,Wolfable,thewolfable.com,Growing 23-person Ahmedabad creative and perfo...,Connect with leadership on LinkedIn to discuss...
4,Brand Height,brandheight.com,Boutique 8-member agency in Ahmedabad speciali...,Message the founders on LinkedIn proposing an ...
5,Thanksweb Marketing Pvt Ltd,thanksweb.com,36-employee Ahmedabad digital growth agency ma...,Email their operations team offering a quick w...
6,Flora Fountain,florafountain.com,Ahmedabad boutique agency managing 150+ brand ...,Send a personalized email to the founders demo...
7,Kleverish,kleverish.com,Performance marketing agency in Ahmedabad gene...,Reach out via LinkedIn to discuss automated le...
8,Digihify,digihify.com,Ahmedabad boutique performance agency scaling ...,Message agency leaders on LinkedIn offering a ...
9,DC Technolabs,dctechnolabs.com,Under-50 employee digital agency in Ahmedabad ...,Send an email pitch showing how an automated C...


## Step 7 — Export the leads to a file you can open in Excel / Google Sheets

In [ ]:
from datetime import datetime

filename = f"leads_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
leads_df.to_csv(filename, index=False)
print(f"✅ Saved {len(leads_df)} leads to {filename}")
print("Download it from the Files panel (folder icon) on the left sidebar,")
print("or run the next cell to download it directly to your laptop.")

✅ Saved 10 leads to leads_20260819_1719.csv
Download it from the Files panel (folder icon) on the left sidebar,
or run the next cell to download it directly to your laptop.


In [ ]:
# Optional: trigger a direct download to your laptop
from google.colab import files
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>